# MACD(12,26,9) on 2W BTC — does a bullish cross after a bearish-cross-near-ATH mark the bottom?

**Setup** — BTC/USD daily spot from Bitstamp (`data/bitstamp_btcusd_daily.json`, 2011-08 → 2026-05) resampled to fixed 2-week candles (anchored Mon 2011-08-22 UTC). Standard MACD on close: EMA(12) − EMA(26), signal = EMA(9) of MACD.

**Event chain** — for each new 2W ATH (close > all prior 2W closes), find the next *bearish* MACD cross within `N_ATH` bars. From that bearish cross, find the next *bullish* MACD cross. Take the lowest 2W low between them as the cycle low; measure how far above it the bullish-cross close prints.

**Bottom call** — the bullish cross is said to have *marked the bottom* if its close is within `PCT_BAND` of the cycle low.

Tune `N_ATH` and `PCT_BAND` in the setup cell. ~383 2W bars covers the 2011, 2013, 2017, 2021×2, 2024, and 2025 cycles.

In [ ]:
import sys, json
from pathlib import Path
from datetime import datetime, timezone
import numpy as np

_p = Path.cwd()
while _p != _p.parent and not (_p / 'jplus').is_dir():
    _p = _p.parent
REPO_ROOT = _p
BITSTAMP_FILE = REPO_ROOT / 'data' / 'bitstamp_btcusd_daily.json'

# Bar size: 7 = weekly, 14 = bi-weekly. Both anchored to Monday 2011-08-22.
BAR_DAYS  = 7
BAR_SECS  = BAR_DAYS * 86400
ANCHOR_DT = datetime(2011, 8, 22, tzinfo=timezone.utc)  # Monday
ANCHOR_TS = int(ANCHOR_DT.timestamp())

# MACD
MACD_FAST, MACD_SLOW, MACD_SIG = 12, 26, 9

# Event params — keep the ATH-lookback ≈ 3 months regardless of bar size.
N_ATH    = max(1, round(84 / BAR_DAYS))   # ≈ 12 weeks: 12 on 1W, 6 on 2W
PCT_BAND = 0.10                            # bull cross 'marks bottom' if close within ±10% of cycle low

print(f'Source: {BITSTAMP_FILE.name}')
print(f'Bar: {BAR_DAYS}d  Anchor: {ANCHOR_DT.date()}  N_ATH={N_ATH} bars (~{N_ATH*BAR_DAYS}d)  band=±{PCT_BAND*100:.0f}%')

In [ ]:
# Load Bitstamp daily BTC/USD, resample to BAR_DAYS-day candles (fixed anchored buckets)
with open(BITSTAMP_FILE, 'r', encoding='utf-8') as f:
    raw = json.load(f)
raw.sort(key=lambda r: r['ts'])

buckets = {}  # bucket_id -> [first_ts, open, high, low, close, n_days]
for r in raw:
    ts = int(r['ts'])
    if ts < ANCHOR_TS:
        continue
    b = (ts - ANCHOR_TS) // BAR_SECS
    if b not in buckets:
        buckets[b] = [ts, r['open'], r['high'], r['low'], r['close'], 1]
    else:
        cur = buckets[b]
        cur[2] = max(cur[2], r['high'])
        cur[3] = min(cur[3], r['low'])
        cur[4] = r['close']
        cur[5] += 1

# Drop the trailing partial bar
max_b = max(buckets)
if buckets[max_b][5] < BAR_DAYS:
    del buckets[max_b]

keys = sorted(buckets)
n = len(keys)
bar_open_ts = np.array([ANCHOR_TS + k * BAR_SECS for k in keys], dtype=np.int64)
open_  = np.array([buckets[k][1] for k in keys], dtype=float)
high   = np.array([buckets[k][2] for k in keys], dtype=float)
low    = np.array([buckets[k][3] for k in keys], dtype=float)
close  = np.array([buckets[k][4] for k in keys], dtype=float)
n_days = np.array([buckets[k][5] for k in keys], dtype=int)

short_bars = int((n_days < BAR_DAYS).sum())
print(f'{BAR_DAYS}d bars: {n}  from {datetime.fromtimestamp(int(bar_open_ts[0]), tz=timezone.utc).date()} to {datetime.fromtimestamp(int(bar_open_ts[-1]), tz=timezone.utc).date()}')
print(f'bars with <{BAR_DAYS} daily rows (gaps): {short_bars}')
print(f'First 3 closes: {close[:3].round(2)}   Last 3 closes: {close[-3:].round(0)}')

In [ ]:
def ema(x, period):
    """Standard EMA, seeded from first value (TV convention)."""
    k = 2.0 / (period + 1.0)
    out = np.full_like(x, np.nan, dtype=float)
    out[0] = x[0]
    for i in range(1, len(x)):
        out[i] = x[i] * k + out[i-1] * (1 - k)
    return out

ema_fast = ema(close, MACD_FAST)
ema_slow = ema(close, MACD_SLOW)
macd_ln  = ema_fast - ema_slow
signal   = ema(macd_ln, MACD_SIG)
hist     = macd_ln - signal

# Crossovers — use sign change of (macd - signal). Warmup = MACD_SLOW + MACD_SIG bars.
warmup = MACD_SLOW + MACD_SIG
bull_x = np.zeros(n, dtype=bool)
bear_x = np.zeros(n, dtype=bool)
for i in range(warmup, n):
    if hist[i-1] <= 0 < hist[i]:
        bull_x[i] = True
    elif hist[i-1] >= 0 > hist[i]:
        bear_x[i] = True

print(f'warmup bars: {warmup}  bullish crosses: {bull_x.sum()}  bearish crosses: {bear_x.sum()}')

In [ ]:
# Walk the series: detect new ATHs (close-based), then find qualifying bearish->bullish cycles.
ath_close = np.maximum.accumulate(close)
is_new_ath = np.concatenate(([True], close[1:] > ath_close[:-1]))
ath_idx = np.where(is_new_ath)[0]

# For each ATH, find first bearish cross within N_ATH bars
qualifying_bear = []   # list of (ath_i, bear_i)
seen_bear = set()
for a in ath_idx:
    if a < warmup:
        continue
    for j in range(a, min(a + N_ATH + 1, n)):
        if bear_x[j] and j not in seen_bear:
            qualifying_bear.append((a, j))
            seen_bear.add(j)
            break

# For each qualifying bearish, find next bullish cross
cycles = []  # list of dicts
for ath_i, bear_i in qualifying_bear:
    bull_i = None
    for j in range(bear_i + 1, n):
        if bull_x[j]:
            bull_i = j; break
    if bull_i is None:
        cycles.append(dict(ath_i=ath_i, bear_i=bear_i, bull_i=None, status='open'))
        continue
    seg_low_i  = bear_i + np.argmin(low[bear_i:bull_i+1])
    cycle_low  = low[seg_low_i]
    cycle_low_close = close[seg_low_i]
    dist_pct  = (close[bull_i] - cycle_low) / cycle_low
    marked    = abs(dist_pct) <= PCT_BAND
    cycles.append(dict(
        ath_i=ath_i, bear_i=bear_i, bull_i=bull_i, low_i=seg_low_i,
        ath_close=close[ath_i], bear_close=close[bear_i], bull_close=close[bull_i],
        cycle_low=cycle_low, cycle_low_close=cycle_low_close,
        bars_bear_to_bull=bull_i - bear_i,
        bars_bull_after_low=bull_i - seg_low_i,
        dist_pct=dist_pct, marked=bool(marked), status='closed',
    ))

print(f'qualifying bearish crosses (within {N_ATH} bars of an ATH): {len(qualifying_bear)}')
print(f'closed cycles: {sum(1 for c in cycles if c["status"]=="closed")}   open: {sum(1 for c in cycles if c["status"]=="open")}')

In [ ]:
def fmt_date(ts):
    return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime('%Y-%m-%d')

print(f'=== Qualifying cycles (ATH → bearish ≤ {N_ATH} bars → bullish) ===')
print(f'  band for "marked bottom" = bull-cross close within ±{PCT_BAND*100:.0f}% of cycle low\n')
hdr = f'  {"#":>2} {"ATH":>11} {"ath$":>9} {"bear":>11} {"bear$":>9} {"low":>11} {"low$":>9} {"bull":>11} {"bull$":>9} {"Δbars":>5} {"low→bull":>8} {"dist%":>7} {"bottom?":>8}'
print(hdr)
print('  ' + '-' * (len(hdr) - 2))
for k, c in enumerate(cycles, 1):
    ath_dt  = fmt_date(bar_open_ts[c['ath_i']])
    bear_dt = fmt_date(bar_open_ts[c['bear_i']])
    if c['status'] == 'open':
        print(f'  {k:>2} {ath_dt:>11} {close[c["ath_i"]]:>9.0f} {bear_dt:>11} {close[c["bear_i"]]:>9.0f} {"":>11} {"":>9} {"OPEN":>11} {"":>9} {"":>5} {"":>8} {"":>7} {"":>8}')
        continue
    low_dt  = fmt_date(bar_open_ts[c['low_i']])
    bull_dt = fmt_date(bar_open_ts[c['bull_i']])
    flag = 'YES' if c['marked'] else 'no'
    print(f'  {k:>2} {ath_dt:>11} {c["ath_close"]:>9.0f} {bear_dt:>11} {c["bear_close"]:>9.0f} {low_dt:>11} {c["cycle_low"]:>9.0f} {bull_dt:>11} {c["bull_close"]:>9.0f} {c["bars_bear_to_bull"]:>5d} {c["bars_bull_after_low"]:>8d} {c["dist_pct"]*100:>+6.1f}% {flag:>8}')

closed = [c for c in cycles if c['status'] == 'closed']
if closed:
    hits = sum(1 for c in closed if c['marked'])
    dists = np.array([c['dist_pct'] for c in closed])
    print(f'\n  hits: {hits}/{len(closed)} = {hits/len(closed)*100:.0f}%   median dist above low: {np.median(dists)*100:+.1f}%   mean: {dists.mean()*100:+.1f}%')
    print(f'  bull cross lag past cycle low ({BAR_DAYS}d bars): median={int(np.median([c["bars_bull_after_low"] for c in closed]))}, max={max(c["bars_bull_after_low"] for c in closed)}')

In [ ]:
# Sensitivity: how does the hit-rate change with the band threshold?
closed = [c for c in cycles if c['status'] == 'closed']
if closed:
    dists = np.array([abs(c['dist_pct']) for c in closed])
    print(f'{"band":>6} {"hits":>5} {"rate":>7}')
    for band in (0.05, 0.10, 0.15, 0.20, 0.30, 0.50):
        hits = int((dists <= band).sum())
        print(f'{band*100:>5.0f}% {hits:>5d} {hits/len(closed)*100:>6.0f}%')
else:
    print('no closed cycles yet')

In [ ]:
# === Did the bull cross actually end the bear cycle? ===
# Two tests, increasing strictness:
#   STRICT  — price never dips ≤ cycle_low between bull cross and the NEXT bear MACD
#             cross (when MACD flips back). Mostly tautological — MACD bullish ≈
#             rising price — but rules out instant whipsaws.
#   ULTIMATE — price never dips ≤ cycle_low between bull cross and the NEXT NEW ATH
#             being set (the moment the prior bear cycle is unambiguously closed).
#             If still pending at end of data, marked 'open'.

prior_ath = np.maximum.accumulate(close)  # close at index i = ATH up to & including i

print('=== Did the bull cross end the bear cycle? ===')
hdr = f'  {"#":>2} {"bull":>11} {"cyc_low$":>9} {"strict":>7} {"strict_peak":>11}  {"ultimate":>9} {"new_ATH_date":>13} {"min_low_in_seg$":>15}'
print(hdr)
print('  ' + '-' * (len(hdr) - 2))

strict_holds, strict_fails = 0, 0
ult_holds, ult_fails, ult_open = 0, 0, 0
for k, c in enumerate(cycles, 1):
    if c['status'] != 'closed':
        continue
    bull_i = c['bull_i']
    cyc_lo = c['cycle_low']

    # STRICT: until next bear cross
    next_bear = next((j for j in range(bull_i + 1, n) if bear_x[j]), None)
    end_s = next_bear if next_bear is not None else n - 1
    strict_low  = low[bull_i+1 : end_s+1].min() if end_s > bull_i else low[bull_i]
    strict_high = close[bull_i+1 : end_s+1].max() if end_s > bull_i else close[bull_i]
    strict_peak = (strict_high - c['bull_close']) / c['bull_close']
    strict_hold = strict_low > cyc_lo
    strict_holds += strict_hold; strict_fails += (not strict_hold)

    # ULTIMATE: until a new ATH is set after the bull cross
    new_ath_i = None
    for j in range(bull_i + 1, n):
        if close[j] > prior_ath[bull_i]:
            new_ath_i = j; break
    if new_ath_i is None:
        ult_verdict = 'open'; new_ath_dt = '(none)'; ult_min = low[bull_i+1:].min() if bull_i+1 < n else low[bull_i]
        ult_open += 1
    else:
        ult_min = low[bull_i+1 : new_ath_i+1].min()
        ult_hold = ult_min > cyc_lo
        ult_verdict = 'hold' if ult_hold else 'FAIL'
        new_ath_dt  = fmt_date(bar_open_ts[new_ath_i])
        if ult_hold: ult_holds += 1
        else:        ult_fails += 1

    print(f'  {k:>2} {fmt_date(bar_open_ts[bull_i]):>11} {cyc_lo:>9.0f} '
          f'{("hold" if strict_hold else "FAIL"):>7} {strict_peak*100:>+10.1f}%  '
          f'{ult_verdict:>9} {new_ath_dt:>13} {ult_min:>15.0f}')

total_closed = strict_holds + strict_fails
total_ult    = ult_holds + ult_fails
print(f'\n  STRICT (until next bear cross):  hold {strict_holds}/{total_closed} = {strict_holds/total_closed*100:.0f}%')
if total_ult:
    print(f'  ULTIMATE (until next new ATH):   hold {ult_holds}/{total_ult} = {ult_holds/total_ult*100:.0f}%   fail {ult_fails}   ({ult_open} still open / never made new ATH)')

In [ ]:
# === Upside after the bull cross: how far does price run before the next new ATH? ===
# For each closed cycle, measure two gains from the bull-cross close:
#   gain_to_new_ATH  — close on the first 2W bar that exceeds prior_ath (the moment
#                      the prior cycle's high is breached). Cycles where the bull
#                      cross itself prints the new ATH show ~0%.
#   peak_to_cycle_top — max close reached anywhere between the bull cross and the
#                       NEXT qualifying bear-after-ATH event (or end of data).

print('=== Gain from bull cross to next new ATH ===\n')
hdr = f'  {"#":>2} {"bull":>11} {"bull$":>9} {"prior_ATH$":>10} {"new_ATH date":>13} {"new_ATH$":>9} {"gain_to_new":>11}  {"peak date":>11} {"peak$":>9} {"peak_gain":>10}'
print(hdr)
print('  ' + '-' * (len(hdr) - 2))

new_ath_gains   = []
peak_gains      = []
days_to_new_ath = []

for k, c in enumerate(cycles, 1):
    if c['status'] != 'closed':
        continue
    bull_i = c['bull_i']
    bull_px = c['bull_close']
    pa = float(prior_ath[bull_i])

    # gain to first new ATH after bull cross
    new_ath_i = next((j for j in range(bull_i + 1, n) if close[j] > pa), None)

    # peak close until the next qualifying bear (or end of data)
    next_qual_bear = None
    for j in range(bull_i + 1, n):
        if any(j == cc.get('bear_i') for cc in cycles[k:]):
            next_qual_bear = j; break
    end_i = next_qual_bear if next_qual_bear is not None else n - 1
    peak_seg = close[bull_i+1 : end_i+1] if end_i > bull_i else close[bull_i:bull_i+1]
    peak_i_rel = int(np.argmax(peak_seg))
    peak_i = bull_i + 1 + peak_i_rel if end_i > bull_i else bull_i
    peak_px = float(peak_seg[peak_i_rel])
    peak_gain = (peak_px - bull_px) / bull_px

    if new_ath_i is not None:
        new_ath_px = float(close[new_ath_i])
        g = (new_ath_px - bull_px) / bull_px
        days = (int(bar_open_ts[new_ath_i]) - int(bar_open_ts[bull_i])) // 86400
        new_ath_gains.append(g)
        days_to_new_ath.append(days)
        gain_str = f'{g*100:>+10.1f}%'
        new_ath_dt = fmt_date(bar_open_ts[new_ath_i])
        new_ath_str = f'{new_ath_px:>9.0f}'
    else:
        gain_str = f'{"(none)":>11}'; new_ath_dt = '(none)'; new_ath_str = f'{"":>9}'

    peak_gains.append(peak_gain)
    print(f'  {k:>2} {fmt_date(bar_open_ts[bull_i]):>11} {bull_px:>9.0f} {pa:>10.0f} '
          f'{new_ath_dt:>13} {new_ath_str} {gain_str}  '
          f'{fmt_date(bar_open_ts[peak_i]):>11} {peak_px:>9.0f} {peak_gain*100:>+9.1f}%')

if new_ath_gains:
    ng = np.array(new_ath_gains)
    print(f'\n  gain to next new ATH:   n={len(ng)}  mean={ng.mean()*100:+.1f}%   median={np.median(ng)*100:+.1f}%   min={ng.min()*100:+.1f}%   max={ng.max()*100:+.1f}%')
    print(f'  days bull→new ATH:      median={int(np.median(days_to_new_ath))}d   range={min(days_to_new_ath)}-{max(days_to_new_ath)}d')
if peak_gains:
    pg = np.array(peak_gains)
    print(f'  peak gain to cycle top: n={len(pg)}  mean={pg.mean()*100:+.1f}%   median={np.median(pg)*100:+.1f}%   min={pg.min()*100:+.1f}%   max={pg.max()*100:+.1f}%')

In [ ]:
# Optional visual: BTC 2W close (log) with markers + MACD/signal panel
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

dts = [datetime.fromtimestamp(int(t), tz=timezone.utc) for t in bar_open_ts]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

ax1.semilogy(dts, close, lw=1.2, color='#222')
ax1.set_ylabel('BTC close (log)')
ax1.set_title(f'BTC 2W — qualifying cycles (N_ATH={N_ATH}, band=\u00b1{PCT_BAND*100:.0f}%)')

for c in cycles:
    ax1.plot(dts[c['ath_i']],  close[c['ath_i']],  marker='^', ms=10, color='#1f77b4', mew=0)
    ax1.plot(dts[c['bear_i']], close[c['bear_i']], marker='v', ms=10, color='#d62728', mew=0)
    if c['status'] == 'closed':
        ax1.plot(dts[c['low_i']],  low[c['low_i']],     marker='o', ms=8,  color='#2ca02c', mew=0)
        col = '#2ca02c' if c['marked'] else '#ff7f0e'
        ax1.plot(dts[c['bull_i']], close[c['bull_i']],  marker='*', ms=14, color=col,       mew=0)

ax2.plot(dts, macd_ln, color='#1f77b4', lw=1.0, label='MACD')
ax2.plot(dts, signal,  color='#d62728', lw=1.0, label='signal')
ax2.bar(dts, hist, width=10, color=np.where(hist >= 0, '#9fd49f', '#f0a0a0'), edgecolor='none')
ax2.axhline(0, color='#555', lw=0.5)
ax2.legend(loc='upper left', fontsize=8)
ax2.set_ylabel('MACD(12,26,9)')

ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

print('legend: \u25b2 ATH (close)  \u25bc bear-cross  \u25cf cycle low  \u2605 bull-cross (green = "bottom", orange = late)')

## Reading the output

- **`dist%`** — the bullish-cross close minus the cycle low, divided by the cycle low. `+5%` means the bullish cross fired 5% above the lowest 2W low in the bear leg, i.e. it confirmed the bottom early. `+40%` means the rebound was already well underway by the time MACD crossed up.
- **`low→bull`** — number of 2W bars between the cycle-low bar and the bullish-cross bar. MACD is lagging by construction, so this is almost always positive; small values mean a tight confirmation, large values mean MACD waited a long time.
- **Hit rate** — fraction of closed cycles where `|dist%| ≤ PCT_BAND`. Read the per-cycle rows; the aggregate hides regime variation across the 2013/2017/2021/2024 cycles.
- **Open cycles** — a qualifying bearish cross with no subsequent bullish cross yet. Watch for it in the live MACD panel.

**Caveats** — 2W candles are anchored to Mon 2011-08-22 UTC; TradingView's biweekly alignment may differ by one week and shift crossover timing by ±1 bar. Two single-day gaps in Bitstamp daily history (2019-04-07, 2022-01-02) — each affects at most one 2W bar's OHLC and is negligible. Earliest Bitstamp data has flat OHLC (low-volume venue) so MACD signals before ~2013 are unreliable; the 35-bar warmup absorbs most of that.